<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Ai/blob/main/Image_classifier_DNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import numpy as np

In [44]:
def init_parameter(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l], layers_dims[l-1]) * np.sqrt(2 / layers_dims[l-1])
    b[l] = np.zeros((layers_dims[l], 1))

  return W , b

In [45]:
def output(W , b , X , L):
  A = X
  activation = [A]
  Zs = []
  def relu(z):
    return np.maximum(0,z)
  def sigmoid(z):
    return 1/(1+np.exp(-z))

  for l in range(1, L-1):
    A_prev = A
    Z = np.dot(W[l],A_prev)+b[l]
    Zs.append(Z)

    A = relu(Z)
    activation.append(A)

  ZL = np.dot(W[L-1],A)+b[L-1]
  Zs.append(ZL)

  AL = sigmoid(ZL)
  return activation , Zs , AL

In [46]:
import numpy as np

def loss(Y_true, AL_pred):
  m = Y_true.shape[1]
  epsilon = 1e-8
  AL_pred_clipped = np.clip(AL_pred, epsilon, 1 - epsilon)
  cost = -(1/m) * np.sum(Y_true * np.log(AL_pred_clipped) + (1 - Y_true) * np.log(1 - AL_pred_clipped))
  return cost

In [47]:
def backward(W, b, activations, Zs, AL, Y_true):

    m = Y_true.shape[1]

    L = len(W)

    dW = {}
    db = {}

    dZ = AL - Y_true


    dW[L] = (1/m) * np.dot(dZ, activations[L-1].T)
    db[L] = (1/m) * np.sum(dZ, axis=1, keepdims=True)


    dA = np.dot(W[L].T, dZ)

    for l in reversed(range(1, L)):

        dZ = np.array(dA, copy=True)

        dZ[Zs[l-1] <= 0] = 0

        dW[l] = (1/m) * np.dot(dZ, activations[l-1].T)
        db[l] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

        if l > 1:
            dA = np.dot(W[l].T, dZ)

    return dW, db

In [48]:
def gradient_descent(W , b , dW , db , lr):

  for l in W.keys():
    W[l] = W[l] - lr * dW[l]
    b[l] = b[l] - lr * db[l]
  return W , b

In [49]:
!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset

Dataset URL: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset
License(s): apache-2.0
dog-and-cat-classification-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [50]:
# import zipfile
# import os

# zip_file_path = '/content/dog-and-cat-classification-dataset.zip'
# extraction_path = '/content/dataset'

# # Create the extraction directory if it doesn't exist
# os.makedirs(extraction_path, exist_ok=True)

# with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#     zip_ref.extractall(extraction_path)

# print(f"Extracted contents to: {extraction_path}")
# print(os.listdir(extraction_path))

In [51]:
import os

# Use the extraction_path defined in the previous cell
os.listdir(os.path.join(extraction_path, "PetImages"))

['Dog', 'Cat']

In [52]:
cats = len(os.listdir(os.path.join(extraction_path, "PetImages/Cat")))
dogs = len(os.listdir(os.path.join(extraction_path, "PetImages/Dog")))

print(cats)
print(dogs)

12499
12499


In [53]:
from PIL import Image

img = Image.open(os.path.join(extraction_path, "PetImages/Cat/0.jpg"))

In [54]:
img_array = np.array(img)

print(img_array.shape)

(375, 500, 3)


In [55]:
img = img.resize((64,64))

img_array = np.array(img)
print(img_array.shape)
img_array = img_array/255
print(img_array)

(64, 64, 3)
[[[0.80784314 0.65490196 0.35294118]
  [0.83529412 0.68235294 0.38039216]
  [0.86666667 0.70588235 0.39607843]
  ...
  [0.97254902 0.84313725 0.54117647]
  [0.96470588 0.81960784 0.50196078]
  [0.94901961 0.79607843 0.48235294]]

 [[0.81176471 0.65882353 0.35686275]
  [0.83529412 0.68235294 0.38039216]
  [0.8627451  0.70588235 0.39607843]
  ...
  [0.97254902 0.85098039 0.55294118]
  [0.96470588 0.83137255 0.5254902 ]
  [0.95686275 0.80784314 0.49411765]]

 [[0.81568627 0.65882353 0.36078431]
  [0.83921569 0.68235294 0.38039216]
  [0.8627451  0.70196078 0.39607843]
  ...
  [0.96862745 0.85098039 0.54901961]
  [0.96470588 0.84313725 0.5372549 ]
  [0.96470588 0.82352941 0.51372549]]

 ...

 [[0.63921569 0.49803922 0.23137255]
  [0.65098039 0.51372549 0.24705882]
  [0.67058824 0.5254902  0.24705882]
  ...
  [0.01176471 0.01568627 0.        ]
  [0.01176471 0.01568627 0.        ]
  [0.01176471 0.01568627 0.        ]]

 [[0.62352941 0.49019608 0.22745098]
  [0.63529412 0.50196078 

In [56]:
X = []
y = []

In [57]:
for i in os.listdir(os.path.join(extraction_path, "PetImages/Cat")):
  img = Image.open(os.path.join(extraction_path, "PetImages/Cat", i))
  img = img.convert("RGB")
  img = img.resize((64, 64))
  img = np.array(img)
  img_array = np.array(img)
  img_array = img_array/255
  X.append(img_array)
  y.append(0)

In [58]:
for i in os.listdir(os.path.join(extraction_path, "PetImages/Dog")):
  img = Image.open(os.path.join(extraction_path, "PetImages/Dog", i))
  img = img.convert("RGB")
  img = img.resize((64, 64))
  img = np.array(img)
  img_array = np.array(img)
  img_array = img_array/255
  X.append(img_array)
  y.append(1)

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [59]:
X = np.array(X)
Y = np.array(y)
print(X.shape)
print(Y.shape)

(24998, 64, 64, 3)
(24998,)


In [60]:
perm = np.random.permutation(X.shape[0])

X = X[perm]
Y = Y[perm]

In [61]:
X = X.reshape(X.shape[0], -1)
print(X.shape)
print(X.shape)

(24998, 12288)
(24998, 12288)


In [62]:
Y = Y.reshape(1, -1)
print(Y.shape)

(1, 24998)


In [63]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y.flatten(),
    test_size=0.2,
    random_state=42
)

X_train = X_train.T
X_test = X_test.T
Y_train = Y_train.reshape(1, -1)
Y_test = Y_test.reshape(1, -1)

In [64]:
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape )

(12288, 19998)
(12288, 5000)
(1, 19998)
(1, 5000)


In [65]:
print(X_train)

print(Y_train.shape)

[[0.83137255 0.3372549  0.38039216 ... 0.39607843 0.03137255 0.56470588]
 [0.83529412 0.41568627 0.38823529 ... 0.38431373 0.1254902  0.52941176]
 [0.91372549 0.22745098 0.39215686 ... 0.31764706 0.12156863 0.20392157]
 ...
 [0.07058824 0.85882353 0.63137255 ... 0.35294118 0.04705882 0.61568627]
 [0.05882353 0.80392157 0.64705882 ... 0.34901961 0.1372549  0.54117647]
 [0.1254902  0.69803922 0.70196078 ... 0.29019608 0.12156863 0.18823529]]
(1, 19998)


In [66]:
def model(dim, X_data, Y_data):
  W ,b = init_parameter(dim)
  for i in range(500):
    L = len(dim)
    activation , Zs , AL = output(W , b , X_data , L)
    cost = loss(Y_data , AL)
    dW , db = backward(W , b , activation , Zs , AL , Y_data)
    # print(dW)
    W , b = gradient_descent(W , b , dW , db , 0.1)
    if i % 50 == 0:
      print(f"Iteration {i}: Cost = {cost}")
  return W , b

dim = [12288 , 64 , 32 , 1]

W, b = model(dim, X_train, Y_train)

Iteration 0: Cost = 0.7057573167613228
Iteration 50: Cost = 0.681469160934909
Iteration 100: Cost = 0.6710881984172902
Iteration 150: Cost = 0.6633808362784808
Iteration 200: Cost = 0.6570180058240048
Iteration 250: Cost = 0.6523396807107156
Iteration 300: Cost = 0.6485954193844289
Iteration 350: Cost = 0.6448334957052478
Iteration 400: Cost = 0.6421829854890251
Iteration 450: Cost = 0.6390963808702076


In [67]:
activation , Zs , AL = output(W , b , X_test , len(dim))
pred = (AL > 0.5)


In [68]:
accuracy = np.mean(pred == Y_test)
print("Model Accuracy: ",accuracy*100,"%")

Model Accuracy:  61.78 %


In [71]:
for i in os.listdir(os.path.join(extraction_path, "PetImages/Cat")):
  img = Image.open(os.path.join(extraction_path, "PetImages/Cat", i))
  img = img.convert("RGB")
  img = img.resize((64, 64))
  img = np.array(img)
  img_array = np.array(img)
  img_array = img_array/255
  X.append(img_array)
  y.append(0)

AttributeError: 'numpy.ndarray' object has no attribute 'append'

In [ ]:
for i in os.listdir(os.path.join(extraction_path, "PetImages/Dog")):
  img = Image.open(os.path.join(extraction_path, "PetImages/Dog", i))
  img = img.convert("RGB")
  img = img.resize((64, 64))
  img = np.array(img)
  img_array = np.array(img)
  img_array = img_array/255
  X.append(img_array)
  y.append(1)

In [ ]:
X = np.array(X)
Y = np.array(y)
print(X.shape)
print(Y.shape)

In [ ]:
perm = np.random.permutation(X.shape[0])

X = X[perm]
Y = Y[perm]

In [ ]:
X = X.reshape(X.shape[0], -1)
print(X.shape)
print(X.shape)

In [ ]:
Y = Y.reshape(1, -1)
print(Y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y.flatten(),
    test_size=0.2,
    random_state=42
)

X_train = X_train.T
X_test = X_test.T
Y_train = Y_train.reshape(1, -1)
Y_test = Y_test.reshape(1, -1)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape )

In [ ]:
print(X_train)

print(Y_train.shape)

In [ ]:
def model(dim, X_data, Y_data):
  W ,b = init_parameter(dim)
  for i in range(500):
    L = len(dim)
    activation , Zs , AL = output(W , b , X_data , L)
    cost = loss(Y_data , AL)
    dW , db = backward(W , b , activation , Zs , AL , Y_data)
    # print(dW)
    W , b = gradient_descent(W , b , dW , db , 0.1)
    if i % 50 == 0:
      print(f"Iteration {i}: Cost = {cost}")
  return W , b

dim = [12288 , 64 , 32 , 1]

W, b = model(dim, X_train, Y_train)

In [ ]:
activation , Zs , AL = output(W , b , X_test , len(dim))
pred = (AL > 0.5)


In [ ]:
accuracy = np.mean(pred == Y_test)
print("Model Accuracy: ",accuracy*100,"%")

In [ ]:
for i in os.listdir(os.path.join(extraction_path, "PetImages/Cat")):
  img = Image.open(os.path.join(extraction_path, "PetImages/Cat", i))
  img = img.convert("RGB")
  img = img.resize((64, 64))
  img = np.array(img)
  img_array = np.array(img)
  img_array = img_array/255
  X.append(img_array)
  y.append(0)

In [ ]:
for i in os.listdir(os.path.join(extraction_path, "PetImages/Dog")):
  img = Image.open(os.path.join(extraction_path, "PetImages/Dog", i))
  img = img.convert("RGB")
  img = img.resize((64, 64))
  img = np.array(img)
  img_array = np.array(img)
  img_array = img_array/255
  X.append(img_array)
  y.append(1)

In [ ]:
X = np.array(X)
Y = np.array(y)
print(X.shape)
print(Y.shape)

In [ ]:
perm = np.random.permutation(X.shape[0])

X = X[perm]
Y = Y[perm]

In [ ]:
X = X.reshape(X.shape[0], -1)
print(X.shape)
print(X.shape)

In [ ]:
Y = Y.reshape(1, -1)
print(Y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y.flatten(),
    test_size=0.2,
    random_state=42
)

X_train = X_train.T
X_test = X_test.T
Y_train = Y_train.reshape(1, -1)
Y_test = Y_test.reshape(1, -1)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape )

Input Layer      : 12288 neurons


Hidden Layer 1   : 128 neurons (ReLU)


Hidden Layer 2   : 64 neurons (ReLU)


Hidden Layer 3   : 32 neurons (ReLU)


Output Layer     : 1 neuron (Sigmoid)

In [ ]:
print(X_train)

print(Y_train.shape)

In [ ]:
def model(dim, X_data, Y_data):
  W ,b = init_parameter(dim)
  for i in range(500):
    L = len(dim)
    activation , Zs , AL = output(W , b , X_data , L)
    cost = loss(Y_data , AL)
    dW , db = backward(W , b , activation , Zs , AL , Y_data)
    # print(dW)
    W , b = gradient_descent(W , b , dW , db , 0.1)
    if i % 50 == 0:
      print(f"Iteration {i}: Cost = {cost}")
  return W , b

dim = [12288 , 64 , 32 , 1]

W, b = model(dim, X_train, Y_train)

In [ ]:
activation , Zs , AL = output(W , b , X_test , len(dim))
pred = (AL > 0.5)



In [ ]:
accuracy = np.mean(pred == Y_test)
print("Model Accuracy: ",accuracy*100,"%")